# Milestone  — Validate Gold Outputs

In this notebook, we validate all Gold tables to ensure they are reliable for business use.
Checks include:

- Row counts
- Null dimensions
- Duplicate grains
- Impossible metric values
- Sample data inspection

In [0]:
# Cell 1 — Configuration & Table Names
# Define catalog, schemas, and Gold table names
import yaml
from pyspark.sql import functions as F



catalog = "vattenfall_dev"
gold_schema = "gold"

gold_tables = {
    "gold_daily_market_summary": f"{catalog}.{gold_schema}.gold_daily_market_summary",
    "gold_weather_impact_summary": f"{catalog}.{gold_schema}.gold_weather_impact_summary",
    "gold_grid_incident_summary": f"{catalog}.{gold_schema}.gold_grid_events",
    "gold_regional_operations_dashboard": f"{catalog}.{gold_schema}.gold_regional_operations_dashboard"
}

# Optional: define expected grains for duplicate checks
expected_grains = {
    "gold_daily_market_summary": ["report_day", "region"],
    "gold_weather_impact_summary": ["report_day", "region"],
    "gold_grid_incident_summary": ["event_day", "region", "severity_band"],
    "gold_regional_operations_dashboard": ["report_day", "region"]
}

In [0]:
# Cell 2 — Logging function
# Simple function to print validation info

def log_info(table_name, rows=None, nulls=None, duplicates=None, notes=None):
    print(f"Table: {table_name}")
    if rows is not None:
        print(f"Row count: {rows}")
    if nulls:
        print("Null dimension counts:")
        for col, count in nulls.items():
            print(f"  {col}: {count}")
    if duplicates is not None:
        print(f"Duplicate grain rows: {duplicates}")
    if notes:
        print(f"Notes: {notes}")
    print("-" * 60)

In [0]:
# Cell 3 — Validation loop
# Iterate over all Gold tables and validate

for table_name, table_path in gold_tables.items():
    try:
        df = spark.table(table_path)
        rows = df.count()
        
        # Null checks on grain columns
        null_counts = {}
        for col in expected_grains.get(table_name, []):
            null_counts[col] = df.filter(F.col(col).isNull()).count()
        
        # Duplicate grain check
        duplicates = df.groupBy(*expected_grains.get(table_name, [])).count().filter("count > 1").count()
        
        # Impossible values check (basic examples)
        impossible_notes = []
        if table_name == "gold_daily_market_summary":
            impossible = df.filter(F.col("avg_market_price") < 0).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have negative avg_market_price")
        elif table_name == "gold_weather_impact_summary":
            impossible = df.filter(F.col("avg_temperature_c") < -50).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have avg_temperature_c < -50°C")
        elif table_name == "gold_grid_incident_summary":
            impossible = df.filter(F.col("incident_count") < 0).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have negative incident_count")
        elif table_name == "gold_regional_operations_dashboard":
            impossible = df.filter(F.col("avg_market_price") < 0).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have negative avg_market_price")
        
        # Log results
        log_info(table_name, rows=rows, nulls=null_counts, duplicates=duplicates, notes=impossible_notes)
        
        # Show top 5 rows for inspection
        print("Sample rows:")
        df.show(5)
        
    except Exception as e:
        print(f"Error validating {table_name}: {e}")
        print("-" * 60)

In [0]:
# Cell 3 — Validation loop
# Iterate over all Gold tables and validate

for table_name, table_path in gold_tables.items():
    try:
        df = spark.table(table_path)
        rows = df.count()
        
        # Null checks on grain columns
        null_counts = {}
        for col in expected_grains.get(table_name, []):
            null_counts[col] = df.filter(F.col(col).isNull()).count()
        
        # Duplicate grain check
        duplicates = df.groupBy(*expected_grains.get(table_name, [])).count().filter("count > 1").count()
        
        # Impossible values check (basic examples)
        impossible_notes = []
        if table_name == "gold_daily_market_summary":
            impossible = df.filter(F.col("avg_market_price") < 0).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have negative avg_market_price")
        elif table_name == "gold_weather_impact_summary":
            impossible = df.filter(F.col("avg_temperature_c") < -50).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have avg_temperature_c < -50°C")
        elif table_name == "gold_grid_incident_summary":
            impossible = df.filter(F.col("incident_count") < 0).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have negative incident_count")
        elif table_name == "gold_regional_operations_dashboard":
            impossible = df.filter(F.col("avg_market_price") < 0).count()
            if impossible > 0:
                impossible_notes.append(f"{impossible} rows have negative avg_market_price")
        
        # Log results
        log_info(table_name, rows=rows, nulls=null_counts, duplicates=duplicates, notes=impossible_notes)
        
        # Show top 5 rows for inspection
        print("Sample rows:")
        df.show(5)
        
    except Exception as e:
        print(f"Error validating {table_name}: {e}")
        print("-" * 60)